### Transformer upgrade (HF 404 client error)

In [1]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

# Import Necessary Libraries

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
import librosa

from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    TrainingArguments,
    TrainerState,
    TrainerControl,
    EarlyStoppingCallback,
    pipeline
)

import pandas as pd
import numpy as np
import csv
import librosa
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from sklearn.model_selection import train_test_split
import random

2025-11-16 00:29:24.287507: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763252964.462085      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763252964.511993      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import transformers
print(f"Transformers version: {transformers.__version__}")
print(f"Torch version: {torch.__version__}")
print(f"Librosa version: {librosa.__version__}")

Transformers version: 4.57.1
Torch version: 2.6.0+cu124
Librosa version: 0.11.0


### Reproducibility

In [4]:
def set_seed(seed):
    """random seeds for reproducibility."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if multi-GPU.
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

### Memory log (to avoid OOM) 

In [5]:
def log_memory(stage: str):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_alloc = torch.cuda.max_memory_allocated() / 1024**3
        print(f"[MEMORY {stage}] Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB | Max Alloc: {max_alloc:.2f} GB")
    else:
        print(f"[MEMORY {stage}] CPU mode - no GPU stats available.")

# Necessary CONFIG

In [6]:
MODEL_ID = "bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium"
CACHE_DIR = "./hf_cache"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dir = Path("/kaggle/input/shobdotori/Train")
test_dir = Path("/kaggle/input/shobdotori/Test")
train_annot = Path("/kaggle/input/shobdotori/Train_annotation")
output_dir = "/kaggle/working/results/checkpoint-final"

In [7]:
# Function to clean RAM & vRAM
import gc
import ctypes

def cln_memory():
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)
    torch.cuda.empty_cache()

# Analysis and Preprocessing

### Regions

In [8]:
import os
from IPython.display import Audio

# Define the base directory for the training data
train_data_path = '/kaggle/input/shobdotori/Train'

# Get a list of all region directories
regions = [name for name in os.listdir(train_data_path) if os.path.isdir(os.path.join(train_data_path, name))]

print("Available Regions:")
for i, region in enumerate(regions, start=1):
    print(f"{i}. {region}")

Available Regions:
1. Mymensingh
2. Kushtia
3. Lakshmipur
4. Pabna
5. Dhaka
6. Brahmanbaria
7. Jessore
8. Sylhet
9. Jhenaidah
10. Khulna
11. Bogura
12. Noakhali
13. Rajshahi
14. Natore
15. Barisal
16. Comilla
17. Feni
18. Rangpur
19. Chittagong
20. Bhola


In [9]:
import os
from IPython.display import Audio

# Choose a region
selected_region = 'Bhola'

# Path to the selected region's audio files and transcription
region_path = os.path.join(train_data_path, selected_region)
annot_csv_filepath = os.path.join(train_annot, f"{selected_region}.csv")


# List all audio files in the selected region
audio_files = [os.path.join(region_path, f) for f in os.listdir(region_path) if f.endswith('.wav')]

if audio_files:
    # Play the first audio file found in that region
    first_audio_file = audio_files[0]
    #annotation of the audio
    print(f"Playing audio from: {first_audio_file}")
    print(f"Annotation file path: {annot_csv_filepath}")
    df = pd.read_csv(annot_csv_filepath)

    # Get the base name of the audio file to match with the 'audio' column in df
    first_audio_filename = os.path.basename(first_audio_file)

    # Find the transcription for the first audio file
    transcription_for_first_audio = df[df['audio'] == first_audio_filename]['text'].iloc[0]

    print(f"Transcription for {first_audio_filename}: {transcription_for_first_audio}")
    display(Audio(first_audio_file))
else:
    print(f"No WAV audio files found in the '{selected_region}' region.")

Playing audio from: /kaggle/input/shobdotori/Train/Bhola/male_bhola_121.wav
Annotation file path: /kaggle/input/shobdotori/Train_annotation/Bhola.csv
Transcription for male_bhola_121.wav: আজ রাতে আমাদের অতিথি আসবে।


In [10]:
class ASRDataset(Dataset):
    def __init__(self, audio_paths, transcriptions, processor):
        self.audio_paths = audio_paths
        self.transcriptions = transcriptions
        self.processor = processor

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        # 4a. Load and resample audio
        audio_filepath = self.audio_paths[idx]
        audio, sr = librosa.load(audio_filepath, sr=16000)

        # 4b. Extract audio features
        input_features = self.processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt').input_features

        # 4c. Tokenize transcription
        transcription = self.transcriptions[idx]
        labels = self.processor.tokenizer(transcription, max_length=255, truncation=True, return_tensors='pt').input_ids

        # 4d. Squeeze dimensions
        input_features = input_features.squeeze(0)
        labels = labels.squeeze(0)

        # 4e. Return a dictionary
        return {
            "input_features": input_features,
            "labels": labels
        }

print("ASRDataset class defined.")

ASRDataset class defined.


In [11]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
processor = AutoProcessor.from_pretrained("bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium")
model = AutoModelForSpeechSeq2Seq.from_pretrained("bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium")
model.to(DEVICE)
model.config.forced_decoder_ids = None

print("Bengaliai-asr processor and model loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

Bengaliai-asr processor and model loaded successfully.


In [12]:
all_audio_paths = []
all_transcriptions = []

for filename in os.listdir(train_annot):
    if filename.endswith('.csv'):
        region = filename.replace('.csv', '')
        csv_path = os.path.join(train_annot, filename)
        df = pd.read_csv(csv_path)
        for index, row in df.iterrows():
            audio = row['audio']
            text = row['text']
            audio_path = os.path.join(train_dir, region, audio)
            if os.path.exists(audio_path):
                all_audio_paths.append(audio_path)
                all_transcriptions.append(text)

train_audio_paths, eval_audio_paths, train_transcriptions, eval_transcriptions = train_test_split(
    all_audio_paths, all_transcriptions, test_size=0.1, random_state=42
)

# Create instances of ASRDataset
train_dataset = ASRDataset(train_audio_paths, train_transcriptions, processor)
eval_dataset = ASRDataset(eval_audio_paths, eval_transcriptions, processor)

print(f"Number of training audio files found: {len(train_audio_paths)}")
print(f"Number of training transcriptions found: {len(train_transcriptions)}")
print(f"Number of evaluation audio files found: {len(eval_audio_paths)}")
print(f"Number of evaluation transcriptions found: {len(eval_transcriptions)}")
print("ASRDatasets for training and evaluation created successfully.")

Number of training audio files found: 3015
Number of training transcriptions found: 3015
Number of evaluation audio files found: 335
Number of evaluation transcriptions found: 335
ASRDatasets for training and evaluation created successfully.


# Check model if it works

In [13]:
# # --- Use a small subset for a quick test run ---
# small_audio_paths = all_audio_paths[:50]  # Using only the first 100 samples
# small_transcriptions = all_transcriptions[:50]

# train_audio_paths, eval_audio_paths, train_transcriptions, eval_transcriptions = train_test_split(
#     small_audio_paths, small_transcriptions, test_size=0.1, random_state=42
# )

# # Create instances of ASRDataset
# train_dataset = ASRDataset(train_audio_paths, train_transcriptions, processor)
# eval_dataset = ASRDataset(eval_audio_paths, eval_transcriptions, processor)

# print("--- RUNNING IN DEBUG MODE WITH A SMALL DATASET ---")
# print(f"Number of training audio files found: {len(train_audio_paths)}")
# print(f"Number of training transcriptions found: {len(train_transcriptions)}")
# print(f"Number of evaluation audio files found: {len(eval_audio_paths)}")
# print(f"Number of evaluation transcriptions found: {len(eval_transcriptions)}")
# print("ASRDatasets for training and evaluation created successfully.")

## Training

In [14]:
# !pip install -U flash-attn --no-build-isolation

In [15]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # Tweak PyTorch's Memory Allocator

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_steps=50,                  # CHANGED: Increased for a longer run
    num_train_epochs=2,               # CHANGED: Use epochs instead of max_steps
    #gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",            # Renamed from evaluation_strategy to eval_strategy
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,                   # CHANGED: Evaluate and save less frequently
    eval_steps=50,                   # CHANGED: Evaluate and save less frequently
    logging_steps=25,                 # CHANGED: Log less frequently
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    #save_total_limit=2, #save only best checkpoint
    optim="adafactor", #optimize memory allocatoion
    dataloader_num_workers=2,
    torch_compile=True
)

print("Seq2SeqTrainingArguments configured for full dataset training.")

The speedups for torchdynamo mostly come with GPU Ampere or higher and which is not detected here.


Seq2SeqTrainingArguments configured for full dataset training.


In [16]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

print("DataCollatorSpeechSeq2SeqWithPadding class defined.")

DataCollatorSpeechSeq2SeqWithPadding class defined.


In [17]:
!pip install jiwer
!pip install evaluate

import evaluate

# Load the WER metric
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    # Get label_ids (ground truth) and predictions (logits)
    label_ids = pred.label_ids
    predictions = pred.predictions

    # Replace -100 in label_ids with the pad_token_id to allow decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode label_ids to text (ground truth transcriptions)
    decoded_labels = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True);

    # Decode predictions to text
    decoded_predictions = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)

    # Calculate WER
    wer = wer_metric.compute(predictions=decoded_predictions, references=decoded_labels)

    return {"wer": wer}

print("compute_metrics function defined and WER metric loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 37.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires google-cloud-bigquery[bqstorage,panda

compute_metrics function defined and WER metric loaded.


In [18]:
from transformers import TrainerCallback, TrainingArguments, TrainerState, TrainerControl

class MemoryCleaningCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        log_memory(f"After training step {state.global_step}")
        cln_memory()

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"Epoch {state.epoch} finished.")
        log_memory("After epoch end")
        cln_memory()

    def on_train_end(self, args, state, control, **kwargs):
        print("Training finished.")
        log_memory("After training end")
        cln_memory()

In [19]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[MemoryCleaningCallback(),
               EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting model training...")
trainer.train()
print("Model training completed.")
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

cln_memory()

Starting model training...


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


[MEMORY After training step 1] Allocated: 2.90 GB | Reserved: 12.93 GB | Max Alloc: 11.44 GB


Step,Training Loss,Validation Loss,Wer
50,0.350100,0.252379,0.282840
100,0.135000,0.131543,0.177753
150,0.061400,0.097877,0.146451


[MEMORY After training step 2] Allocated: 2.90 GB | Reserved: 12.97 GB | Max Alloc: 11.46 GB
[MEMORY After training step 3] Allocated: 2.90 GB | Reserved: 12.95 GB | Max Alloc: 11.46 GB
[MEMORY After training step 4] Allocated: 2.90 GB | Reserved: 12.98 GB | Max Alloc: 11.46 GB
[MEMORY After training step 5] Allocated: 2.90 GB | Reserved: 12.95 GB | Max Alloc: 11.46 GB
[MEMORY After training step 6] Allocated: 2.90 GB | Reserved: 12.96 GB | Max Alloc: 11.46 GB
[MEMORY After training step 7] Allocated: 2.90 GB | Reserved: 12.95 GB | Max Alloc: 11.46 GB
[MEMORY After training step 8] Allocated: 2.90 GB | Reserved: 13.01 GB | Max Alloc: 11.46 GB
[MEMORY After training step 9] Allocated: 2.90 GB | Reserved: 12.94 GB | Max Alloc: 11.46 GB
[MEMORY After training step 10] Allocated: 2.90 GB | Reserved: 13.03 GB | Max Alloc: 11.46 GB
[MEMORY After training step 11] Allocated: 2.90 GB | Reserved: 12.95 GB | Max Alloc: 11.46 GB
[MEMORY After training step 12] Allocated: 2.90 GB | Reserved: 12.96

...

[MEMORY After training step 101] Allocated: 2.91 GB | Reserved: 12.25 GB | Max Alloc: 11.47 GB
[MEMORY After training step 102] Allocated: 2.91 GB | Reserved: 12.90 GB | Max Alloc: 11.47 GB
[MEMORY After training step 103] Allocated: 2.91 GB | Reserved: 13.01 GB | Max Alloc: 11.47 GB
[MEMORY After training step 104] Allocated: 2.91 GB | Reserved: 13.00 GB | Max Alloc: 11.47 GB
[MEMORY After training step 105] Allocated: 2.91 GB | Reserved: 12.96 GB | Max Alloc: 11.47 GB
[MEMORY After training step 106] Allocated: 2.91 GB | Reserved: 12.99 GB | Max Alloc: 11.47 GB
[MEMORY After training step 107] Allocated: 2.91 GB | Reserved: 13.01 GB | Max Alloc: 11.47 GB
[MEMORY After training step 108] Allocated: 2.91 GB | Reserved: 12.99 GB | Max Alloc: 11.47 GB
[MEMORY After training step 109] Allocated: 2.91 GB | Reserved: 12.94 GB | Max Alloc: 11.47 GB
[MEMORY After training step 110] Allocated: 2.91 GB | Reserved: 12.97 GB | Max Alloc: 11.47 GB
[MEMORY After training step 111] Allocated: 2.91 G

You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


[MEMORY After training step 151] Allocated: 2.91 GB | Reserved: 12.26 GB | Max Alloc: 11.47 GB
[MEMORY After training step 152] Allocated: 2.91 GB | Reserved: 12.91 GB | Max Alloc: 11.47 GB
[MEMORY After training step 153] Allocated: 2.91 GB | Reserved: 12.97 GB | Max Alloc: 11.47 GB
[MEMORY After training step 154] Allocated: 2.91 GB | Reserved: 12.99 GB | Max Alloc: 11.47 GB
[MEMORY After training step 155] Allocated: 2.91 GB | Reserved: 12.94 GB | Max Alloc: 11.47 GB
[MEMORY After training step 156] Allocated: 2.91 GB | Reserved: 12.99 GB | Max Alloc: 11.47 GB
[MEMORY After training step 157] Allocated: 2.91 GB | Reserved: 12.96 GB | Max Alloc: 11.47 GB
[MEMORY After training step 158] Allocated: 2.91 GB | Reserved: 12.99 GB | Max Alloc: 11.47 GB
[MEMORY After training step 159] Allocated: 2.91 GB | Reserved: 12.94 GB | Max Alloc: 11.47 GB
[MEMORY After training step 160] Allocated: 2.91 GB | Reserved: 12.98 GB | Max Alloc: 11.47 GB
[MEMORY After training step 161] Allocated: 2.91 G

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Training finished.
[MEMORY After training end] Allocated: 2.89 GB | Reserved: 3.11 GB | Max Alloc: 11.47 GB
Model training completed.


In [20]:
MODEL_ID = "/kaggle/working/results"
print(f"Does {MODEL_ID} exist? {os.path.exists(MODEL_ID)}")
if os.path.exists(MODEL_ID):
    print(f"Files in {MODEL_ID}: {os.listdir(MODEL_ID)}")
    # If checkpoints were saved, they might be in subdirs like 'checkpoint-10' or 'checkpoint-20'
    for subdir in os.listdir(MODEL_ID):
        full_subdir = os.path.join(MODEL_ID, subdir)
        if os.path.isdir(full_subdir) and 'checkpoint' in subdir:
            print(f"Files in {full_subdir}: {os.listdir(full_subdir)}")
else:
    print(f"Directory {MODEL_ID} does not exist. Training may not have saved any checkpoints.")

Does /kaggle/working/results exist? True
Files in /kaggle/working/results: ['checkpoint-190', 'checkpoint-50', 'checkpoint-150', 'checkpoint-100', 'checkpoint-final']
Files in /kaggle/working/results/checkpoint-190: ['rng_state.pth', 'training_args.bin', 'optimizer.pt', 'trainer_state.json', 'generation_config.json', 'model.safetensors', 'scaler.pt', 'config.json', 'scheduler.pt']
Files in /kaggle/working/results/checkpoint-50: ['rng_state.pth', 'training_args.bin', 'optimizer.pt', 'trainer_state.json', 'generation_config.json', 'model.safetensors', 'scaler.pt', 'config.json', 'scheduler.pt']
Files in /kaggle/working/results/checkpoint-150: ['rng_state.pth', 'training_args.bin', 'optimizer.pt', 'trainer_state.json', 'generation_config.json', 'model.safetensors', 'scaler.pt', 'config.json', 'scheduler.pt']
Files in /kaggle/working/results/checkpoint-100: ['rng_state.pth', 'training_args.bin', 'optimizer.pt', 'trainer_state.json', 'generation_config.json', 'model.safetensors', 'scaler.pt

In [21]:
# # After training, in your training cell:
# output_dir = "/kaggle/working/results/checkpoint-final"

# # Save model + processor
# model.save_pretrained(output_dir)
# processor.save_pretrained(output_dir)   # <-- THIS IS CRITICAL

# print(f"Model and processor saved to {output_dir}")

# Punctuation Models

In [22]:
# from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer

# PUNCT_MODELS = [
#     '/kaggle/input/punctuation-models-jibanananda-das/Punctuation_Models/punct-model-6layers/',
#     '/kaggle/input/punctuation-models-jibanananda-das/Punctuation_Models/punct-model-8layers/',
#     '/kaggle/input/punctuation-models-jibanananda-das/Punctuation_Models/punct-model-11layers/',
#     '/kaggle/input/punctuation-models-jibanananda-das/Punctuation_Models/punct-model-12layers/'
# ]

# PUNCT_WEIGHTS = [[1.0, 1.4, 1.0, 0.8]]
# models = [AutoModelForTokenClassification.from_pretrained(f).eval().cuda() for f in PUNCT_MODELS]
# tokenizer = AutoTokenizer.from_pretrained(PUNCT_MODELS[0])

# def add_punctuation(text):
#     input_ids = tokenizer(text).input_ids
#     with torch.no_grad():
#         model = models[0]
#         logits = torch.nn.functional.softmax(
#             model(input_ids=torch.LongTensor([input_ids]).cuda()).logits[0, 1:-1],
#             dim=1).cpu()
#         for model in models[1:]:
#             logits += torch.nn.functional.softmax(
#                 model(input_ids=torch.LongTensor([input_ids]).cuda()).logits[0, 1:-1],
#                 dim=1).cpu()
#         logits = logits / len(models)
#         logits *= torch.FloatTensor(PUNCT_WEIGHTS)
#         label_ids = torch.argmax(logits, dim=-1)

#         tokens = tokenizer(text, add_special_tokens=False).input_ids
#         punct_text = ""
#         for index, token in enumerate(tokens):
#             token_str = tokenizer.decode(token)
#             if '##' not in token_str:
#                 punct_text += " " + token_str
#                 pass
#             else:
#                 punct_text += token_str[2:]
#             punct_text += ['', '।', ',', '?'][label_ids[index].item()]

#     punct_text = punct_text.strip()
#     return punct_text

In [23]:
!pip install bnunicodenormalizer

In [24]:
from bnunicodenormalizer import Normalizer 

norm = Normalizer(allow_english=True)
def normalize_sentence(sentence):
    words = sentence.split()
    normalized_words = [norm(word)['normalized'] or word for word in words]
    return ' '.join(normalized_words)

# Submission

In [25]:
cln_memory()

# Ensure the model and processor are loaded and on the correct device
MODEL_ID = "/kaggle/working/results/checkpoint-final"  # Point to the checkpoint subdir
CACHE_DIR = "./hf_cache"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the fine-tuned processor and model from the checkpoint

processor = WhisperProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model.to(DEVICE)
print("Fine-tuned Whisper model and processor loaded successfully.")


def transcribe(audio_path: str) -> str:
    """
    Transcribes an audio file using the fine-tuned Whisper model.
    """
    try:
        # 1. Load and resample the audio file
        audio, sr = librosa.load(audio_path, sr=16000)
        # 2. Process the audio to get input features
        input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
        # 3. Move input features to the appropriate device (GPU or CPU)
        input_features = input_features.to(DEVICE)
        # 4. Generate token IDs
        predicted_ids = model.generate(input_features)
        # 5. Decode the token IDs to text
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        # ---------- NORMALIZER ----------
        transcription = normalize_sentence(transcription)
        
        return transcription.strip()
    except Exception as e:
        print(f"Error transcribing {audio_path}: {e}")
        return ""
    finally:
        cln_memory()  # Clean up after each transcription to avoid OOM on large test sets

if __name__ == "__main__":
    # Define the path to the test directory
    test_dir = Path("/kaggle/input/shobdotori/Test")
    if not test_dir.exists():
        print(f"Test directory not found: {test_dir}")
    else:
        submission_path = "submission.csv"
        wav_files = sorted(test_dir.glob("*.wav"))
        print(f"Found {len(wav_files)} test files to transcribe.")
        # Open the submission file for writing
        with open(submission_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
            writer.writerow(["audio", "text"])  # Write the header
            # Transcribe each audio file and write to the CSV
            for wav_file in wav_files:
                txt = transcribe(str(wav_file))
                writer.writerow([wav_file.name, txt])
                print(f"{wav_file.name} -> {txt}")
        print(f"\nSubmission saved to {submission_path}")

Fine-tuned Whisper model and processor loaded successfully.
Found 450 test files to transcribe.


`generation_config` default values have been modified to match model-specific defaults: {'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


test_001.wav -> তুমি কি খেলতে যাবে?
test_002.wav -> তুমি কি আমাকে কলমটা দেবে?
test_003.wav -> আজ দুপুরে রাস্তায় পানি জমেছিল।
test_004.wav -> আজকের সকালে হঠাৎ বৃষ্টি নেমেছিল।
test_005.wav -> তুমি কি ফটো তুলেছো?
test_006.wav -> তুমি কি ফোন দেবে?
test_007.wav -> আজকের দাবাত মেঘ জমে আছে।
test_008.wav -> আমার ছোট ভাই স্কুলে যাচ্ছে।
test_009.wav -> আমি গান লিখেছি আজ।
test_010.wav -> তুমি কি আজ বন্ধুর সাথে দেখা করেছো?
test_011.wav -> দরজাটা ধীরে ধীরে বন্ধ করে দিল।
test_012.wav -> আমি আজ নতুন জুতো কিনেছি।
test_013.wav -> তুমি কি আজ অফিসে যাবে?
test_014.wav -> আমি সকালের নাস্তায় ডিম খেয়েছি।
test_015.wav -> আজ দুপুরে রাস্তায় দুপুরে ভিজেছিল।
test_016.wav -> আমি বিকেলে হাঁটতে যেতে চাই।
test_017.wav -> আমাদের স্কুলে আজ বিকেল অনুষ্ঠান আসবে।
test_018.wav -> আমি বাজার থেকে আম এনেছি।
test_019.wav -> তুমি কি কখনো কবিতা লিখেছো?
test_020.wav -> আমি নতুন ফোনের সাথে চার্জার কিনেছি।
test_021.wav -> তুমি কি আগামীকাল ঢাকায় যাচ্ছো?
test_022.wav -> আজ বিকেলে আমরা একসাথে গল্প করবো।
test_023.wav -> আমি দুপুরে গল্প